# Study 805 — Cokurtosis Premium — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 3894, 'n_rows': 4147, 'spread_bps': -0.15, 't_nw': -0.11, 't_1s': -0.11, 'hi_bps': 7.39, 'lo_bps': 7.54, 'welch_t': -0.06, 'gross_sharpe': -0.03, 'placebo_obs': -0.15, 'placebo_mean': 0.01, 'placebo_sd': 1.131, 'placebo_p': 0.541, 'placebo_sd_from_null': -0.14, 'placebo_draws': 1000, 'era_early_bps': -1.95, 'era_early_t': -1.21, 'era_early_n': 1760, 'era_late_bps': 1.33, 'era_late_t': 0.61, 'era_late_n': 2134, 'timer_1_gross': -0.15, 'timer_1_cost': 2.14, 'timer_1_net': -2.29, 'timer_1_t': -1.61, 'timer_5_gross': -0.15, 'timer_5_cost': 10.14, 'timer_5_net': -10.29, 'timer_5_t': -7.26, 'null_mean_t': 0.03, 'null_sd_t': 0.9, 'null_fire': 0, 'planted_t': 7.2, 'planted_welch': 9.19, 'fingerprint': '357fd262912f'}

## The headline — long-high-cokurt / short-low-cokurt spread

Daily equal-weight top-30% minus bottom-30% cokurtosis-with-the-market spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-cokurt {R['hi_bps']:+.2f} vs low-cokurt {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -0.15 bps/day  NW(10) t = -0.11  one-sample t = -0.11
books         : high-cokurt +7.39 vs low-cokurt +7.54 bps (Welch t = -0.06)
gross Sharpe  : -0.03 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f}")
print(f"observed sits {R['placebo_sd_from_null']:+.2f} sd from the null (dead centre)")

observed -0.15 bps vs placebo mean +0.010 (sd 1.131) -> p = 0.54100
observed sits -0.14 sd from the null (dead centre)


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1760): -1.95 bps  NW t = -1.21
2018-2026 (n=2134): +1.33 bps  NW t = +0.61


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -0.15 -> net -2.29 bps/day (cost 2.14/day, t=-1.61)
5 bps one-way: gross -0.15 -> net -10.29 bps/day (cost 10.14/day, t=-7.26)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted premium.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cokurtosis import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(knob=0.0, seed=805+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (knob=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(knob=0.009, seed=805, n_assets=40, n_days=1500))
print(f"planted (knob=0.009): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (knob=0), 8 seeds: NW t mean +0.27 (sd 0.95), |t|>=2 in 0/8


planted (knob=0.009): NW t = +7.20, Welch t = +9.19


## Verdict

- **Signal — None.** The Fang-Lai systematic-kurtosis premium does **not** replicate on 50 liquid US mega-caps: the long-high-cokurt / short-low-cokurt spread is **-0.15 bps/day** (NW *t* = **-0.11**) — a flat zero, sitting -0.14 sd from a 1,000-permutation placebo null and flipping sign across eras (*t* = -1.21 / +0.61). The 20-seed synthetic control recovers a *planted* premium cleanly (*t* = +7.20, fires on 0/20 nulls), so the flat result is a real absence, not machinery. Survivorship biases the magnitude only upward.
- **Tradability — Mirage.** There is no gross edge to monetise; the costed book loses money (-2.29 bps/day at 1 bp one-way, -10.29 at 5 bps). A paycheck from a zero is a Mirage.